In [ ]:
import h5py as h5
import numpy as np
from scipy.ndimage import zoom
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import glob
import os

# ===========================================
# Define necessary functions for calculation
# ===========================================


def grid_mapping(data,N):
    # Takes the flattened data and maps each value to its corresponding grid coordinate index.
    newdata = np.reshape(data, (N,N,N), order='F')  # Order reshaping by 'fastest' (Fortran ordering)
    return newdata

class Domain:
    """Represents a single 3D grid chunk of simulation"""

    def __init__(self, vars_path, mc_path, domain_tuple):
        self.indices = domain_tuple  # (L,i,j,k)

        # Load the data immediately as object is created
        with h5.File(vars_path, 'r') as f_vars, h5.File(mc_path, 'r') as f_mc:
            self.grid, self.N = self._unpack_grid(f_vars)
            self.g = self._unpack_metric(f_vars,self.N)
            self.Ka = self._unpack_Ka(f_mc,self.N)

        self.tau = np.full((self.N,self.N,self.N), np.inf)

    # Functions with the '_' prefix are internal
    def _unpack_grid(self, file):
        # Unpack the grid coordinates for a file and return them along 
        # with the number of points N along an axis.

        # Unpack grid coordinates
        coords_group = file['GridCoords']['Step000000']  # Cube with N^3 total points
        x = np.unique(coords_group['x'][:])
        y = np.unique(coords_group['y'][:])
        z = np.unique(coords_group['z'][:])
        N = len(x)
        X,Y,Z = np.meshgrid(x,y,z, indexing='ij')  # Tuple of lists which gives every x,y,z position
        
        # Stacks the x,y,z axes s.t. it acts like a vector field, i.e. has shape (N,N,N,3). 
        # Thus, grid_coords[i,j,k] returns the position vector associated with the grid coordinates (i,j,k).
        grid_coords = np.stack((X,Y,Z), axis=-1) 
        return grid_coords,N

    def _unpack_metric(self, file, N):
        # Unpack metric components
        g_group = file['g']['Step000000']  # First timestep of the metric group, with 6 components (eg. 'xx','xy',...)

        g_metric = np.zeros((N,N,N,3,3))  # Initialize metric
        axes = ['x', 'y', 'z']
        for i,a in enumerate(axes):  # enumerate() extracts the index and element as a tuple
            for j,b in enumerate(axes):

                # Sort strings a and b alphabetically then join the strings to form 'ab' 
                # This prevents calling g_group['yx'] which doesn't exist due to symmetry
                key = "".join(sorted([a,b]))

                # Reshape each component to align with grid indices
                grid_map_comp = grid_mapping(g_group[key],N)  

                # Since key is sorted alphabetically, off-diagonal elements are covered.
                # e.g. i=2,j=1 --> (unsorted) 'zy' --> (sorted) 'yz' so g[z,y] = g_yz
                g_metric[..., i,j] = grid_map_comp  # g[i,j,k, a,b] gives the ab component of the metric at gridpoint [i,j,k] 
        return g_metric

    def _unpack_Ka(self, file, N):
        # Unpack Absorbtion Opacity field Ka
        kappa_group = file['AbsorptionOpacity_4_nue']['Step000000']  
        kappa = kappa_group['scalar'][:]

        # Use mapping function to reshape flattened opacity dataset
        Ka = grid_mapping(kappa,N)  
        
        return Ka

def precompute_ds(dom_g,dx,dy,dz):
    # Initialize ds_dict
    ds_dict = {}

    for di in [-1,0,1]:
        for dj in [-1,0,1]:
            for dk in [-1,0,1]:
                if di==0 and dj==0 and dk==0:
                    continue
                # All possible displacement vectors (excluding 0)
                v = np.array([dx*di, dy*dj, dz*dk])

                # Contract with metric; resulting shape is (N,N,N) as we only contract over the (3,3) slice 
                ds_sq = np.einsum('a, ...ab, b ->...', v, dom_g, v)
                
                # Key will be displacement tuple
                ds_dict[(di,dj,dk)] = np.sqrt(ds_sq)
    return ds_dict
                
def solver_BF(grid, Ka, N, tau_old, ng=3):

    """
    Uses Bellman-Ford algorithm to calculate tau_new on the interior of the domain.

    Returns:
        NDArray: Returns tau_new: a (N,N,N) array of values of tau, updated using the old values.
                 
                 Though the ghost zones are not updated here, they are used to update neighboring points.
    """

    tau_new = np.copy(tau_old)  # Initializing tau_new

    # Define finite displacements (grid is uniformly spaced, so we need only calculate one displacement
    dx = grid[1,0,0,0] - grid[0,0,0,0]
    dy = grid[0,1,0,1] - grid[0,0,0,1]
    dz = grid[0,0,1,2] - grid[0,0,0,2]

    # Load in ds_dict
    ds_field = precompute_ds(dom.g,dx,dy,dz)  
    
    # Iterate over all points within interior
    for i in range(ng,N-ng):
        for j in range(ng,N-ng):
            for k in range(ng,N-ng):
                # Placeholder for current minimum
                current_min = np.inf #tau_old[i,j,k]
                # Iterate over all possible displacements.
                # For each displacement, we compare against the current minimum,
                # replacing the current minimum with the smaller of the two
                for di in [-1,0,1]:
                    for dj in [-1,0,1]:
                        for dk in [-1,0,1]:
                            if di==0 and dj==0 and dk==0: # Don't consider 0 displacement
                                continue

                            # Define neighboring tau_old values 
                            neighbor_tau = tau_old[i + di, j + dj, k + dk]

                            # Ignore infinities, since they will not be selected  
                            if neighbor_tau == np.inf:
                                continue
                            Ka_avg = (Ka[i,j,k] + Ka[i + di, j + dj, k + dk])/2  # Load first order dKa
                            ds = ds_field[(di,dj,dk)][i,j,k]  # Load line element for each displacement
                            
                            # Algorithm must minimize the sum of a neighboring tau and ds*Ka_avg in that direction
                            candidate_min = neighbor_tau + ds*Ka_avg
                            if candidate_min < current_min: # Choose the smallest
                                current_min = candidate_min

                # For each point, update the new value of tau
                tau_new[i,j,k] = current_min
    return tau_new

def update_tau(grid, Ka, N, tau_old):

    tau_new = solver_BF(grid,Ka,N,tau_old)

    # Record maximum change for determining convergence
    diff = np.zeros_like(tau_new)  # Array of zeros to hold finite values

    # For finite differences between the taus
    finite_mask = np.isfinite(tau_old) & np.isfinite(tau_new)
    diff[finite_mask] = np.abs(tau_new[finite_mask] - tau_old[finite_mask])

    # Lopsided points are ones that were infinite and are now finite
    lopsided_mask = np.isinf(tau_old) & np.isfinite(tau_new)
    diff[lopsided_mask] = np.abs(tau_new[lopsided_mask] - tau_old[lopsided_mask]) 

    # Points that were infinite and are now as well stay 0

    # So we take the maximum
    max_change = np.max(diff)
    return tau_new,max_change

def get_face_mappings(N, ng=3):
    """ 
    Returns a dictionary mapping each face (e.g. '+x') to:
    
    (Extraction Slice, Injection Slice, Opposite Face (e.g. '-x'))

    Example:
    >>> '-x': ( np.s_[ng : 2*ng, :, :],         np.s_[0 : ng, :, :],         '+x' ),
    """
    return {
        '-x': (np.s_[ng:2*ng,:,:], np.s_[0:ng,:,:], '+x'),
        '+x': (np.s_[N-2*ng:N-ng,:,:], np.s_[N-ng:N,:,:], '-x'),
        '-y': (np.s_[:,ng:2*ng,:], np.s_[:,0:ng,:], '+y'),
        '+y': (np.s_[:,N-2*ng:N-ng,:], np.s_[:,N-ng:N,:], '-y'),
        '-z': (np.s_[:,:,ng:2*ng], np.s_[:,:,0:ng], '+z'),
        '+z': (np.s_[:,:,N-2*ng:N-ng], np.s_[:,:,N-ng:N], '-z')
    }

def extract_peer_ghosts(domain_registry, ng=3):

    """
    Returns dictionary of outgoing ghost zones for all domains.
    Domains are accessed by key domain_tuple and ghost zones by face, e.g. '+x'.

    To be used in conjunction with function deliver_peer_ghosts() to update ghost zones 
    across all domains.
    """

    # Initialize ghost zone dictionary
    ghost_registry = {}

    for domain_tuple,dom in domain_registry.items():
        ghost_registry[domain_tuple] = {}  # For every domain, we create a dictionary with key (L,i,j,k)
        faces = get_face_mappings(dom.N, ng)  # Faces dictionary with slices for each ghost zone

        for face, (extr_slice,_,_) in faces.items():
            # The previous dictionary we created for this domain maps
            # face to its corresponding slice taken from tau
            ghost_registry[domain_tuple][face] = dom.tau[extr_slice]
    return ghost_registry
    
def deliver_peer_ghosts(domain_registry, neighbors_dict, ghost_registry, ng=3):
    """
    Updates peer ghost zones in dom.tau for each dom in domain_registry.
    For a given dom, it checks for neighbors within that level, and for 
    each neighboring domain, it uses the face name to grab the ghost zone 
    belonging to that face, updating dom.tau in the process.
    """

    # Iterate through all domains
    for domain_tuple,dom in domain_registry.items():
        neighbors = neighbors_dict[domain_tuple]  # Unpack neighbors for a given domain
        faces = get_face_mappings(dom.N,ng)  # Unpack face mappings for the domain

        # For each face, we want the injection slice and the opposite face
        # to map the ghost zone from ghost_registry to its place in the domain
        for face, (_,inj_slice, opposite_face) in faces.items():
            neighbor = neighbors[face]  # Neighbor domain_tuple
            if neighbor == 'True Boundary':
                # Boundary should always be zero!
                dom.tau[inj_slice] = 0.0
            elif neighbor == 'Level Jump':
                pass  # Level interpolation is dealt with in the following section.
            elif neighbor is not None:
                # Prevent overlapping ghost zones from overwriting each other by taking an element-wise min.
                dom.tau[inj_slice] = np.minimum(dom.tau[inj_slice], ghost_registry[neighbor][opposite_face])
                pass

# ============================================
# Organize files by domain position and level
# ============================================

# Use Vars files as an anchor for filename pattern, then use the same scheme for MCMoments
anchor_pattern = os.path.join('Data', 'Vars_IntervalB-Lev*-*.h5')
vars_files = glob.glob(anchor_pattern)

# Extract the file's level and domain with this function
def file_lev_domain(filepath):

    """
    Extracts the refinement level and spatial domain indices from a SpEC filename.

    This function isolates the integers from the string and returns them as a 
    tuple, which is used as a dictionary key for the domain registry.

    Args:
        filepath (str): The full relative or absolute path to the .h5 file.

    Returns:
        tuple: A 4-element integer tuple formatted as (Level, x_idx, y_idx, z_idx).

    Examples:
    >>> key = file_lev_domain(C:/Users/.../Data/Vars_IntervalB-Lev0-4.4.6.h5)
    >>> print(key)
    (0,4,4,6)
    """

    # Ex: 'C:/Users/.../Data/Vars_IntervalB-Lev*-*.h5' -> 'Vars_IntervalB-Lev*-*.h5'
    filename = os.path.basename(filepath)

    # Split by '-' to create a list, e.g. ['Vars_IntervalB','Lev*','*.h5']
    parts = filename.split('-') 

    # Extract level
    level_str = parts[1]
    level = int(level_str.replace('Lev',''))  # 'Lev0' -> '0' -> 0

    # Extract domain
    domain_str = parts[2].replace('.h5','')  # Get rid of filetype
    domain_ind = [int(num) for num in domain_str.split('.')]  # Create a list of domain indices

    # Return a tuple (level, X, Y, Z), which will be sorted in the order level, X, Y, Z.
    return (level, domain_ind[0], domain_ind[1], domain_ind[2])

# Sort through the anchor list
sorted_vars_files = sorted(vars_files, key=file_lev_domain)

def get_mc_path(vars_path):

    directory = os.path.dirname(vars_path)
    vars_filename = os.path.basename(vars_path)

    # Replace 'Vars' with 'MCMoments' in filename
    mcmoments_filename = vars_filename.replace('Vars','MCMoments')
    mc_path = os.path.join(directory,mcmoments_filename)  # Create MCMoments path
    
    return mc_path

# -----------------------------------------------------------------------
# Initialize domain_registry: a dictionary for domains and their datasets
# -----------------------------------------------------------------------

def build_domain_registry():
    domain_registry = {}
    # Loop through all vars files to populate dictionary
    for vars_path in sorted_vars_files:
        domain_tuple = file_lev_domain(vars_path)  # Domain keys look like (L,i,j,k)
        mc_path = get_mc_path(vars_path)  # For each vars path, grab the corresponding mc path

        my_domain = Domain(vars_path, mc_path, domain_tuple)  # Load in datasets from each file

        domain_registry[domain_tuple] = my_domain  # Map each domain's index to its dataset

    return domain_registry

domain_registry = build_domain_registry()

# ---------------------------------------------------------------------
# Build a dictionary that identifies neighbors, even in-between levels.
# ---------------------------------------------------------------------

def build_neighbors_dict(domain_registry):
    """
    Takes a list of all active (L,i,j,k) tuples and builds a dictionary
    telling each domain who its peers, parents, and children are and/or flagging boundaries.
    """
    neighbors = {}  # Initialize dictionary

    # Define shifts for each face of a domain
    directions = {
        '-x': (-1,0,0), '+x': (1,0,0),
        '-y': (0,-1,0), '+y': (0,1,0),
        '-z': (0,0,-1), '+z': (0,0,1),
    }

    for dom in domain_registry:
        L, i, j, k = dom
        neighbors[dom] = {}  # Initialize key structure

        # Peers (same level)
        for face_name, (di,dj,dk) in directions.items():
            neighbor_tuple = (L, i + di, j + dj, k + dk)

            if neighbor_tuple in domain_registry:
                # The domain has a neighbor on the same level
                neighbors[dom][face_name] = neighbor_tuple
            elif L == 0:
                # The domain is a true boundary
                neighbors[dom][face_name] = 'True Boundary'
            else:
                # The domain's neighbor is of another level
                neighbors[dom][face_name] = 'Level Jump'

        # Parent (level - 1) and octant
        if L > 0:
            # The rule for parents is: parent_index = (child_index//2) + 2
            parent_tuple = (L - 1, (i//2)+2, (j//2)+2, (k//2)+2)
            # Parent should exist in registry
            neighbors[dom]['parent'] = parent_tuple if parent_tuple in domain_registry else None
            # Find octant using modulo
            neighbors[dom]['octant'] = (i % 2, j % 2, k % 2)
        else:
            neighbors[dom]['parent'] = None
            neighbors[dom]['octant'] = None

        # Children (level + 1)
        neighbors[dom]['children'] = []

        # 3 is the maximum resolution
        if L < 3:
            for dx in (0,1):
                for dy in (0,1):
                    for dz in (0,1):
                        # The rule for children is: child_index = 2*(parent_index - 2) + (0,1)
                        child_i = 2*(i - 2) + dx
                        child_j = 2*(j - 2) + dy
                        child_k = 2*(k - 2) + dz

                        child_tuple = (L + 1, child_i, child_j, child_k)
                        if child_tuple in domain_registry:
                            neighbors[dom]['children'].append(child_tuple)

    return neighbors

neighbors = build_neighbors_dict(domain_registry)

# ================================================
# Handle inter-level ghost-zones and interpolation
# ================================================

def generate_connections(domain_registry, offset=2):
    """
    Scans the domain registry and builds a flat list of boundary interfaces,
    each element of which looks like: {parent, rule_key, children} in the form of a dictionary.
    Using the rules which are defined in get_interface_rules, we get specific
    slicing rules which will be used to deliver ghost zones from a parent level
    to its child and vice-versa.
    """
    connections = []

    # Define the 6 possible boundary rules
    # Format: (axis_name, parent_index, parent_face, child_face, child_index)
    boundary_rules = [
        ('x', 1, '+x', '-x', 0),
        ('x', 6, '-x', '+x', 7),
        ('y', 1, '+y', '-y', 0),
        ('y', 6, '-y', '+y', 7),
        ('z', 1, '+z', '-z', 0),
        ('z', 6, '-z', '+z', 7)
    ]

    for dom_id in domain_registry:
        L,px,py,pz = dom_id

        # Assume only the parent perspective to avoid double counting
        if L == 3:
            continue

        coords = {'x':px, 'y':py, 'z':pz}

        # Check all 6 directions to see if this domain sits on a boundary
        for axis, p_val, p_face, c_face, c_val in boundary_rules:
            if coords[axis] == p_val:
                # This domain is on a boundary
                active_children = []

                for dq1 in (0,1):
                    for dq2 in (0,1):
                        if axis == 'x':
                            cx = c_val
                            cy = 2 * (py - offset) + dq1
                            cz = 2 * (pz - offset) + dq2
                        elif axis == 'y':
                            cx = 2 * (px - offset) + dq1
                            cy = c_val
                            cz = 2 * (pz - offset) + dq2
                        elif axis == 'z':
                            cx = 2 * (px - offset) + dq1
                            cy = 2 * (py - offset) + dq1
                            cz = c_val

                        child_id = (L + 1, cx, cy, cz)

                        if child_id in domain_registry:
                            active_children.append({
                                'id': child_id,
                                'dq1': dq1,
                                'dq2': dq2
                            })

                    if active_children:
                        connections.append({
                            'parent': dom_id,
                            'rule_key': (p_face,c_face),
                            'children': active_children
                        })
    return connections

connections = generate_connections(domain_registry)

def get_interface_rules(N=38, ng=3):
    """
    Generates a dictionary for interface rules, which will be used for
    exchanging ghost zones between levels.
    """

    rules = {
        # Parent is on the Left (+x face) pointing to Child on the Right (-x face)
        ('+x', '-x'): {
            'axis': 0,
            'p_extract_2':  np.s_[N-ng-2 : N-ng,  :, :],  # Extract from parent
            'c_inject_3':   np.s_[0 : ng,         :, :],  # Inject into child
            'c_extract_6':  np.s_[ng : ng+6,      :, :],  # Extract from child
            'p_inject_3':   np.s_[N-ng : N,       :, :],  # Inject into parent
            # Upscaling 2 coarse cells gives 4 fine cells. We need 3. 
            # Since the child is on the right, we keep the right-most 3 cells [1:4]
            'trunc_4_to_3': np.s_[1:4, :, :] 
        },
        
        # Parent is on the Right (-x face) pointing to Child on the Left (+x face)
        ('-x', '+x'): {
            'axis': 0,
            'p_extract_2':  np.s_[ng : ng+2,      :, :],
            'c_inject_3':   np.s_[N-ng : N,       :, :],
            'c_extract_6':  np.s_[N-ng-6 : N-ng,  :, :],
            'p_inject_3':   np.s_[0 : ng,         :, :],
            # Child is on the left, so we keep the left-most 3 cells [0:3]
            'trunc_4_to_3': np.s_[0:3, :, :]
        },

        ('+y', '-y'): {
            'axis': 1,
            'p_extract_2':  np.s_[:, N-ng-2 : N-ng,    :],
            'c_inject_3':   np.s_[:, 0 : ng,           :],
            'c_extract_6':  np.s_[:, ng : ng+6,        :],
            'p_inject_3':   np.s_[:, N-ng : N,         :],
            # Upscaling 2 coarse cells gives 4 fine cells. We need 3. 
            # Since the child is on the right, we keep the right-most 3 cells [1:4]
            'trunc_4_to_3': np.s_[:, 1:4, :] 
        },
        
        ('-y', '+y'): {
            'axis': 1,
            'p_extract_2':  np.s_[:, ng : ng+2,        :],
            'c_inject_3':   np.s_[:, N-ng : N,         :],
            'c_extract_6':  np.s_[:, N-ng-6 : N-ng,    :],
            'p_inject_3':   np.s_[:, 0 : ng,           :],
            # Child is on the left, so we keep the left-most 3 cells [0:3]
            'trunc_4_to_3': np.s_[:, 0:3, :]
        },

        ('+z', '-z'): {
            'axis': 2,
            'p_extract_2':  np.s_[:, :, N-ng-2 : N-ng  ],
            'c_inject_3':   np.s_[:, :, 0 : ng         ],
            'c_extract_6':  np.s_[:, :, ng : ng+6      ],
            'p_inject_3':   np.s_[:, :, N-ng : N       ],
            # Upscaling 2 coarse cells gives 4 fine cells. We need 3. 
            # Since the child is on the right, we keep the right-most 3 cells [1:4]
            'trunc_4_to_3': np.s_[:, :, 1:4] 
        },
        
        ('-z', '+z'): {
            'axis': 2,
            'p_extract_2':  np.s_[:, :, ng : ng+2      ],
            'c_inject_3':   np.s_[:, :, N-ng : N       ],
            'c_extract_6':  np.s_[:, :, N-ng-6 : N-ng  ],
            'p_inject_3':   np.s_[:, :, 0 : ng         ],
            # Child is on the left, so we keep the left-most 3 cells [0:3]
            'trunc_4_to_3': np.s_[:, :, 0:3]
        }
    }
    return rules

def get_quadrant_slice(axis, N=38, dq1=0, dq2=0):
    """
    Generates np.s_ slice for specific quadrant on a face.
    The dqs tell you which quadrant of a face the child domain will be in.
    This will be slicing slabs of shape (2,64,64) and (6,64,64) depending
    on whether a parent is passing to its child or the reverse, respectively.
    """
    start1, end1 = dq1*N, (dq1 + 1)*N
    start2, end2 = dq2*N, (dq2 + 1)*N

    if axis == 0: return np.s_[:, start1:end1, start2:end2]
    if axis == 1: return np.s_[start1:end1, :, start2:end2]
    if axis == 2: return np.s_[start1:end1, start2:end2, :]

def swap_level_ghosts(domain_registry, connections, N=38, ng=3):
    """
    Parent to child ghost zone delivery:

    Extract a slab of shape e.g. (2,38,38) for x-axis, then interpolate to
    higher resolution e.g. (4,76,76). Truncate the 64x64 slice furthest away
    from the child face e.g. (3,76,76). Then use get_quadrant_slice() to assign
    one e.g. (3,76,76) quadrant of this face to the appropriate child.

    --------------------------------------------------------------------------
    
    Child to parent ghost zone delivery:

    Construct a slab of shape e.g. (6,76,76) for x-axis, then use quadrant_slice()
    based on the child's quadrant. Next, we downscale interpolate e.g. (3,38,38) and
    inject this into the parent using interface rules.
    """
    rules = get_interface_rules(N,ng)

    for connection in connections:
        p_dom = domain_registry[connection['parent']]
        rule = rules[connection['rule_key']]  # e.g. ('+x','-x')
        axis = rule[axis]

        # -----------------------------
        # Down-Pass Prep: Parent -> Children
        # -----------------------------
        slab_2 = p_dom.tau[rule['p_extract_2']]  # Get (2,38,38)
        slab_4 = zoom(slab_2, (2.0,2.0,2.0), order=1)  # Turn it into (4,76,76)
        slab_3 = slab_4[rule['trunc_4_to_3']]  # Truncate to (3,76,76)

        # -------------------------------------
        # Prep for Up-Pass: Canvas for Children
        # -------------------------------------
        canvas_shape = [2*N,2*N,2*N]
        canvas_shape[axis] = 6  # e.g. (6,76,76) for x
        stitched_6 = np.full(canvas_shape, np.inf)

        # --------
        # Exchange 
        # --------
        for child in connection['children']:
            c_dom = domain_registry[child['id']]

            # Slice (3,76,76) into the appropriate quadrant for child
            q_slice = get_quadrant_slice(axis, N, child['dq1'], child['dq2'])

            # Down-Pass: Child recieves Parent ghost zone
            c_dom.tau[rule['c_inject_3']] = np.minimum(
                c_dom.tau[rule['c_inject_3']],
                slab_3[q_slice]
            )

            # Up-Pass Prep: Child loads 38x38 face into 76x76 canvas
            stitched_6[q_slice] = c_dom.tau[rule['c_extract_6']]

        # ----------------------------------
        # Up-Pass: Child -> Canvas -> Parent
        # ----------------------------------
        # Downscale the 76x76 canvas back to 38x38
        parent_ghosts = zoom(stitched_6, (0.5,0.5,0.5), order=1)

        # Inject into parent
        p_dom.tau[rule['p_inject_3']] = np.minimum(
            p_dom.tau[rule['p_inject_3']],
            parent_ghosts
        )

# =====================================
# Main Loop - Iterate until convergence
# =====================================

# -------------------------------
# Initialize the true boundaries
# -------------------------------

ghost_dict = extract_peer_ghosts(domain_registry)

deliver_peer_ghosts(domain_registry, neighbors, ghost_dict)

# -----------
# Start loop
# -----------

tolerance = 1e-9
converged = False
iteration = 0
iterations_since_converged = 0

while not converged:
    iteration += 1
    global_max_change = 0.0

    # if iteration == 3:
    #     break
    # ----------------
    # Update interior
    # ----------------

    # Run solver on every domain and track residual
    for domain_tuple,dom in domain_registry.items():
        tau_old = dom.tau
        tau_new, domain_change = update_tau(dom.grid,dom.Ka,dom.N,dom.tau)
        dom.tau = tau_new
        # Update largest residual in the total domain
        if domain_change > global_max_change:
            global_max_change = domain_change

    # -------------------
    # Update ghost zones
    # -------------------

    ghost_dict = extract_peer_ghosts(domain_registry)

    deliver_peer_ghosts(domain_registry, neighbors, ghost_dict)

    swap_level_ghosts(domain_registry, connections)

    # print(f'Iteration: {iteration} | Max Change: {global_max_change}')


    # ----------------------
    # Check for convergence
    # ----------------------

    if global_max_change < tolerance:
        converged = True
        # print([dom.tau[19,19,3:35] for dom in domain_registry.values()])
        # print([dom.Ka[19,19,3:35] for dom in domain_registry.values()])

# taulist = []
# kalist = []
# gridlist = []
# for dom_id, dom in domain_registry.items():
#     taulist.append(dom.tau[3:35,3:35,3:35])
#     kalist.append(dom.Ka[3:35,3:35,3:35])
#     gridlist.append(dom.grid[3:35,3:35,3:35])

# taulist = np.concatenate(taulist,axis=2)
# kalist = np.concatenate(kalist,axis=2)
# gridlist = np.concatenate(gridlist,axis=2)

# x_ind = 16
# Y = gridlist[x_ind,:,0,1]
# Z = gridlist[x_ind,0,:,2]
# tau = taulist[x_ind,:,:].transpose()
# ka = kalist[x_ind,:,:].transpose()

# fig,ax = plt.subplots()
# im = ax.pcolormesh(Y,Z,ka)
# fig.colorbar(im)
# ax.set_xlabel('Y')
# ax.set_ylabel('Z')
# plt.title(f'Ka for domains (4,4,6)&(4,4,7) and x_ind = {x_ind}')

# plt.show()



Iteration: 1 | Max Change: inf
Iteration: 2 | Max Change: inf
Iteration: 3 | Max Change: inf
Iteration: 4 | Max Change: inf
Iteration: 5 | Max Change: inf
Iteration: 6 | Max Change: inf
Iteration: 7 | Max Change: inf
Iteration: 8 | Max Change: inf
Iteration: 9 | Max Change: inf
Iteration: 10 | Max Change: inf
Iteration: 11 | Max Change: inf
Iteration: 12 | Max Change: inf
Iteration: 13 | Max Change: inf
Iteration: 14 | Max Change: inf
Iteration: 15 | Max Change: inf
Iteration: 16 | Max Change: inf
Iteration: 17 | Max Change: 0.0005813518401751269
Iteration: 18 | Max Change: 0.0004810298088510981
Iteration: 19 | Max Change: 0.000413674139230831
Iteration: 20 | Max Change: 0.00040243445311082026
Iteration: 21 | Max Change: 0.00039442171717114
Iteration: 22 | Max Change: 0.00038941247313325524
Iteration: 23 | Max Change: 0.0002797210993696732
Iteration: 24 | Max Change: 0.00027877891630445715
Iteration: 25 | Max Change: 0.00017657951613003157
Iteration: 26 | Max Change: 0.0001310522849596

In [ ]:
taulist = []
gridlist = []
for dom_id, dom in domain_registry.items():
    taulist.append(dom.tau[3:35, 3:35, 3:35])
    gridlist.append(dom.grid[3:35, 3:35, 3:35])

taulist = np.concatenate(taulist, axis=2)
gridlist = np.concatenate(gridlist, axis=2)

# Extract 1D arrays for the axes 
Y = gridlist[0, :, 0, 1]
Z = gridlist[0, 0, :, 2]
num_x_indices = taulist.shape[0]

# --- Plotly Interactive Implementation ---
# 1. Initialize the figure with the first X slice (index 0)
fig = go.Figure(
    data=go.Heatmap(
        x=Y, 
        y=Z, 
        z=taulist[0, :, :].transpose(),
        colorscale='viridis',
        colorbar=dict(title='Tau')
    )
)

# 2. Build the list of steps for the slider
steps = []
for i in range(num_x_indices):
    step = dict(
        method="restyle",
        # We wrap the new z-data in a list because we are updating trace 0
        args=[{"z": [taulist[i, :, :].transpose()]}],
        label=str(i)  # The text that appears on the slider tick
    )
    steps.append(step)

# 3. Create the slider configuration
sliders = [dict(
    active=0,
    currentvalue={"prefix": "X Index: "},
    pad={"t": 50},  # Add top padding so it doesn't overlap the plot
    steps=steps
)]

# 4. Attach the slider and format the layout
fig.update_layout(
    sliders=sliders,
    xaxis_title='Y',
    yaxis_title='Z',
    title='Tau for domains (4,4,6)&(4,4,7)',
    width=800,
    height=600
)

fig.show()

In [ ]:
def funct():
    kalist = []
    gridlist = []
    for dom_id, dom in domain_registry.items():
        kalist.append(dom.Ka[3:35, 3:35, 3:35])
        gridlist.append(dom.grid[3:35, 3:35, 3:35])

    kalist = np.concatenate(kalist, axis=2)
    gridlist = np.concatenate(gridlist, axis=2)

    # Extract 1D arrays for the axes 
    Y = gridlist[0, :, 0, 1]
    Z = gridlist[0, 0, :, 2]
    num_x_indices = kalist.shape[0]

    # --- Plotly Interactive Implementation ---
    # 1. Initialize the figure with the first X slice (index 0)
    fig = go.Figure(
        data=go.Heatmap(
            x=Y, 
            y=Z, 
            z=kalist[0, :, :].transpose(),
            colorscale='viridis',
            colorbar=dict(title='Ka')
        )
    )

    # 2. Build the list of steps for the slider
    steps = []
    for i in range(num_x_indices):
        step = dict(
            method="restyle",
            # We wrap the new z-data in a list because we are updating trace 0
            args=[{"z": [kalist[i, :, :].transpose()]}],
            label=str(i)  # The text that appears on the slider tick
        )
        steps.append(step)

    # 3. Create the slider configuration
    sliders = [dict(
        active=0,
        currentvalue={"prefix": "X Index: "},
        pad={"t": 50},  # Add top padding so it doesn't overlap the plot
        steps=steps
    )]

    # 4. Attach the slider and format the layout
    fig.update_layout(
        sliders=sliders,
        xaxis_title='Y',
        yaxis_title='Z',
        title='Ka for domains (4,4,6)&(4,4,7)',
        width=800,
        height=600
    )


    fig.show()